In [41]:
import pandas as pd
import os

import re

from Bio import SeqIO
import h5py
import json

from tqdm import tqdm
import numpy as np

## prepare mapping between splits and species names

In [42]:
dir = '/mnt/20tb/vsfishman/nn_interpretator/mammals_fasta_dataset/dataset'
metadata_df = pd.read_csv(os.path.join(dir, 'mammals_dataset_metadata3.txt'), sep='\t')
metadata_df

,assembly_accession,organism_name,n_autosomes,number_of_x,number_of_y
0,GCF_004115215.2,Ornithorhynchus anatinus,21,5,20
1,GCF_015852505.1,Tachyglossus aculeatus,31,5,9
2,GCF_902635505.1,Sarcophilus harrisii,6,1,1
3,GCF_011100635.1,Trichosurus vulpecula,9,10,0
4,GCF_030445035.1,Dasypus novemcinctus,46,1,0
...,...,...,...,...,...
120,GCF_002776525.5,Piliocolobus tephrosceles,21,1,1
121,GCF_903992535.2,Arvicola amphibius,17,1,0
122,GCF_019320065.1,Cervus canadensis,33,1,1
123,GCF_028023285.1,Balaenoptera ricei,54,1,0


In [43]:
TEST_SPECIES = ["Sarcophilus harrisii", "Monodelphis domestica", "Manis pentadactyla", "Equus asinus", "Sus scrofa", "Camelus dromedarius", 
        "Phyllostomus discolor", "Myotis daubentonii", "Suncus etruscus", "Lepus europaeus", "Ochotona princeps", 
        "Sciurus carolinensis", "Nycticebus coucang", "Lemur catta", "Cynocephalus volans", 'Elephas maximus indicus', 'Loxodonta africana',
        "Choloepus didactylus", "Ornithorhynchus anatinus", "Tachyglossus aculeatus"]

VALID_SPECIES = ["Lynx canadensis", "Neofelis nebulosa", "Eubalaena glacialis", "Balaenoptera musculus", "Cervus canadensis", "Dama dama",
                "Meriones unguiculatus", "Chionomys nivalis", "Jaculus jaculus", "Callithrix jacchus"]

no_autosomes = ["Mastomys coucha", 'Ursus arctos']

metadata_df = metadata_df[(metadata_df['number_of_x'] != 0) & (metadata_df['number_of_y'] != 0) & (metadata_df['n_autosomes'] != 0)]

TRAIN_SPECIES = metadata_df[metadata_df['organism_name'].apply(lambda x: (x not in VALID_SPECIES) and (x not in TEST_SPECIES))].organism_name.to_list()

In [44]:
split2organism_name = {
    "train": TRAIN_SPECIES,
    "valid": VALID_SPECIES,
    "test": TEST_SPECIES
}

with open('mammals_gender_data/split2organism_name.json', 'w') as f:
    json.dump(split2organism_name, f)

In [45]:
print(len(TRAIN_SPECIES))
print(len(VALID_SPECIES))
print(len(TEST_SPECIES))

28
10
20


# check if all the species have haploid chromosome sets 

In [46]:
def read_chunk_from_hdf5(hdf5_path, dataset_name, start, length):
    with h5py.File(hdf5_path, 'r') as hdf5_file:
        dset = hdf5_file[dataset_name]
        chunk = dset[start:start + length].astype(str)
        return ''.join(chunk)

In [47]:
chr_df = pd.read_csv('mammals_gender_data/mammals_diploid_chr_number.tsv')

In [48]:
for split in ['train', 'valid', 'test']:
    for species in split2organism_name[split]:
        print(species)
        assembly_id = metadata_df[metadata_df['organism_name'] == species]['assembly_accession'].values[0]
        
        with h5py.File(f"mammals_gender_data/{split}.h5", 'r') as hdf5_file:
            autosomes = []
            sex_chromosomes = []
            
            for chr in list(hdf5_file[assembly_id].keys()):
               chr_name = re.findall(r'chromosome\s+([0-9]{1,2}|[A-Za-z]{1,2})', chr)[0]
               if chr_name != "Y" and chr_name != "X":
                   try:
                       autosomes.append(int(chr_name))
                   except:
                       autosomes.append(chr_name)
               else:
                   sex_chromosomes.append(chr_name)

            autosomes = list(set(autosomes))
            sex_chromosomes = list(set(sex_chromosomes))
            autosomes.sort()
            
            print("Sex chromosomes in dataset:", " ".join(sex_chromosomes))
            print("Autosomes in dataset:", autosomes)
            print("Diploid number of chromosomes in dataset:", len(autosomes) * 2 + len(sex_chromosomes))
            print("Diploid number of chromosomes:", chr_df[chr_df['species_name'] == species]['diploid_chr_number'].values[0])
            print(chr_df[chr_df['species_name'] == species]['diploid_chr_number'].values[0] == len(autosomes) * 2 + len(sex_chromosomes))
            print()

Macaca mulatta
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Diploid number of chromosomes in dataset: 42
Diploid number of chromosomes: 42
True

Papio anubis
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Diploid number of chromosomes in dataset: 42
Diploid number of chromosomes: 42
True

Pan paniscus
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Diploid number of chromosomes in dataset: 48
Diploid number of chromosomes: 48
True

Pongo abelii
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Diploid number of chromosomes in dataset: 48
Diploid number of chromosomes: 48
True

Homo sapiens
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4

In [49]:
# metadata_df[metadata_df['organism_name'] == 'Ursus arctos']

In [50]:
# metadata_df[metadata_df['organism_name'] == 'Mastomys coucha']

## Merging mammals, human and mouse data

In [51]:
def create_links(source_file, target_file, source_filename, path="/"):
    """
    Create external links from all groups and datasets in source_file to target_file,
    placing them directly in the root of the target file (no prefixes).
    """
    for key in source_file:
        item = source_file[key]
        target_path = f"{path}{key}"
        if isinstance(item, h5py.Group):
            # Create group in the target and recursevely add links of its subgroups
            if key not in target_file:
                grp = target_file.create_group(key)
            else:
                grp = target_file[key]
            create_links(item, grp, source_filename, path=target_path + "/")
        else:
            target_file[key] = h5py.ExternalLink(source_filename, target_path)


for split in ['train', 'valid', 'test']:
    with h5py.File(f'mammals_gender_data/merged_links_{split}.h5', 'w') as merged_file:

        mammals_file = f'mammals_gender_data/{split}.h5'
        with h5py.File(mammals_file, 'r') as f:
            create_links(f, merged_file, source_filename=mammals_file)

        human_file = f'/mnt/20tb/ykuratov/gender_data_v1/{split}.h5'
        with h5py.File(human_file, 'r') as f:
            create_links(f, merged_file, source_filename=human_file)

        mouse_file = f'/mnt/20tb/ykuratov/mouse_gender_data/{split}.h5'
        with h5py.File(mouse_file, 'r') as f:
            create_links(f, merged_file, source_filename=mouse_file)

    print(f"Merged file 'mammals_gender_data/merged_links_{split}.h5' created.")

OSError: Unable to synchronously create file (unable to truncate a file which is already open)

In [12]:
with h5py.File('mammals_gender_data/merged_links_train.h5', 'r') as f:
    print(len(f.keys()))
    print(f.keys())
    print(f['129P2_OlaHsd']['OlaHsd_chr1'][0:10].astype(str))

81
<KeysViewHDF5 ['129P2_OlaHsd', 'A_J', 'BTBR_T+_Itpr3tf_J', 'BUB_BnJ', 'C3H_HeH', 'C57BL_6NJ', 'C58_J', 'CAST_EiJ', 'CBA_J', 'GCF_000001405.40', 'GCF_000001635.27', 'GCF_000247795.1', 'GCF_002201575.2', 'GCF_002263795.3', 'GCF_002776525.5', 'GCF_003339765.1', 'GCF_008632895.1', 'GCF_008728515.1', 'GCF_009762305.2', 'GCF_009829155.1', 'GCF_011064425.1', 'GCF_011762505.1', 'GCF_011762595.1', 'GCF_014441545.1', 'GCF_016772045.2', 'GCF_023065955.2', 'GCF_023091745.1', 'GCF_024542745.1', 'GCF_025265405.1', 'GCF_028885655.2', 'GCF_029289425.2', 'GCF_030435805.1', 'GCF_032452875.1', 'GCF_036323735.1', 'GCF_902655055.1', 'GCF_922984935.1', 'GCF_947179515.1', 'GCF_949987515.1', 'GCF_963455315.1', 'HG00109', 'HG00118', 'HG00243', 'HG00281', 'HG00371', 'HG00589', 'HG01072', 'HG01501', 'HG01529', 'HG01868', 'HG01946', 'HG01993', 'HG02604', 'HG02966', 'HG02984', 'HG03082', 'HG03129', 'HG03342', 'HG03442', 'HG03446', 'HG03649', 'HG04056', 'HG04177', 'HG04219', 'I_LnJ', 'LP_J', 'MOLF_EiJ', 'NA12873

## Merging metadata

In [13]:
for split in ['train', 'valid', 'test']:
    
    human_metadata_df = pd.read_csv(f'/mnt/20tb/ykuratov/gender_data/{split}.csv', index_col=0)
    mouse_metadata_df = pd.read_csv(f'/mnt/20tb/ykuratov/mouse_gender_data/{split}.csv', index_col=0)

    species = split2organism_name[split]
    mammals_metadata_df = metadata_df[metadata_df['organism_name'].isin(species)]
    mammals_metadata_df = mammals_metadata_df[['organism_name', 'assembly_accession']]
    mammals_metadata_df.rename(columns={'assembly_accession': 'assembly_accession/sample_id'}, inplace=True)
    mammals_metadata_df['sex'] = None
    mammals_metadata_df['diploid'] = False

    human_metadata_df['organism_name'] = 'Homo sapiens'
    human_metadata_df.rename(columns={'sample': 'assembly_accession/sample_id'}, inplace=True)
    human_metadata_df['diploid'] = True
    human_metadata_df['sex'] = human_metadata_df['sex'].map({1: '0', 2: '1'})

    mouse_metadata_df['organism_name'] = 'Mus musculus'
    mouse_metadata_df.rename(columns={'strain_name': 'assembly_accession/sample_id'}, inplace=True)
    mouse_metadata_df.rename(columns={'gender': 'sex'}, inplace=True)
    mouse_metadata_df['sex'] = mouse_metadata_df['sex'].map({'M': 0, 'F': 1})
    mouse_metadata_df['diploid'] = False
    mouse_metadata_df = mouse_metadata_df[['organism_name', 'assembly_accession/sample_id', 'sex', 'diploid']]
    
    all_metadata_df = pd.concat([mammals_metadata_df, human_metadata_df, mouse_metadata_df])
    all_metadata_df.reset_index(drop=True, inplace=True)

    all_metadata_df.to_csv(f'mammals_gender_data/{split}_merged_metadata.csv', index=False)

In [14]:
all_metadata_df

,organism_name,assembly_accession/sample_id,sex,diploid
0,Ornithorhynchus anatinus,GCF_004115215.2,None,False
1,Tachyglossus aculeatus,GCF_015852505.1,None,False
2,Sarcophilus harrisii,GCF_902635505.1,None,False
3,Lemur catta,GCF_020740605.2,None,False
4,Nycticebus coucang,GCF_027406575.1,None,False
5,Loxodonta africana,GCF_030014295.1,None,False
6,Equus asinus,GCF_016077325.2,None,False
7,Sus scrofa,GCF_000003025.6,None,False
8,Camelus dromedarius,GCF_036321535.1,None,False
9,Ochotona princeps,GCF_030435755.1,None,False


In [15]:
for split in ['train', 'valid', 'test']:
    metadata_df = pd.read_csv(f'mammals_gender_data/{split}_merged_metadata.csv')
    metadata_assembly_ids = metadata_df['assembly_accession/sample_id'].unique()

    with h5py.File(f'mammals_gender_data/merged_links_{split}.h5', 'r') as f:
        assembly_ids = list(f.keys())

        assert all([assembly_id in assembly_ids for assembly_id in metadata_assembly_ids])
        print(split, len(metadata_assembly_ids))

train 79
valid 30
test 54


## Checking chromosomes names

In [16]:
for split in ['train', 'valid', 'test']:
    with h5py.File(f'mammals_gender_data/merged_links_{split}.h5', 'r') as f:
        for assembly_id in f.keys():
            for chr in list(f[assembly_id].keys()):
                if "Y" in chr:
                   assert ('chromosome Y' in chr) or ('chrY' in chr)

                if "X" in chr:
                   assert ('chromosome X' in chr) or ('chrX' in chr)
            #    chr_name = re.findall(r'chromosome\s+([0-9]{1,2}|[A-Za-z]{1,2})', chr)[0]


## Checking dataset class

In [16]:
from mammals_gender_dataset import MultiSpeciesGenderDataChunkedDataset

for split in ['train']:
    dataset = MultiSpeciesGenderDataChunkedDataset(data_path='mammals_gender_data', split_name=split, max_n_samples=100, chunk_size=10, n_chunks=10, force_sampling_from_y=True)
    for elem in dataset:
        print(elem)
    

DEBUG:mammals_gender_dataset:Sample organism name: Papio anubis
DEBUG:mammals_gender_dataset:Sample/assembly id: GCF_008728515.1
DEBUG:mammals_gender_dataset:Diploid: False
DEBUG:mammals_gender_dataset:Label: 0
DEBUG:mammals_gender_dataset:Sex chromosomes: ['NC_044997.1 chromosome Y ', 'NC_044996.1 chromosome X ']
DEBUG:mammals_gender_dataset:Autosomes: ['NC_044976.1 chromosome 1 ', 'NC_044977.1 chromosome 2 ', 'NC_044978.1 chromosome 3 ', 'NC_044979.1 chromosome 4 ', 'NC_044980.1 chromosome 5 ', 'NC_044981.1 chromosome 6 ', 'NC_044982.1 chromosome 7 ', 'NC_044983.1 chromosome 8 ', 'NC_044984.1 chromosome 9 ', 'NC_044985.1 chromosome 10 ', 'NC_044986.1 chromosome 11 ', 'NC_044987.1 chromosome 12 ', 'NC_044988.1 chromosome 13 ', 'NC_044989.1 chromosome 14 ', 'NC_044990.1 chromosome 15 ', 'NC_044991.1 chromosome 16 ', 'NC_044992.1 chromosome 17 ', 'NC_044993.1 chromosome 18 ', 'NC_044994.1 chromosome 19 ', 'NC_044995.1 chromosome 20 ']
DEBUG:mammals_gender_dataset:Chromosomes length: {'N

DEBUG:mammals_gender_dataset:Label: 0
DEBUG:mammals_gender_dataset:Sample has Y chromosome: True
DEBUG:mammals_gender_dataset:Y chromosome in sampled chromosomes: True
DEBUG:mammals_gender_dataset:X chromosome in sampled chromosomes: False
DEBUG:mammals_gender_dataset:Sample organism name: Mesoplodon densirostris
DEBUG:mammals_gender_dataset:Sample/assembly id: GCF_025265405.1
DEBUG:mammals_gender_dataset:Diploid: False
DEBUG:mammals_gender_dataset:Label: 1
DEBUG:mammals_gender_dataset:Sex chromosomes: ['NC_082681.1 chromosome X ']
DEBUG:mammals_gender_dataset:Autosomes: ['NC_082661.1 chromosome 1 ', 'NC_082662.1 chromosome 2 ', 'NC_082663.1 chromosome 3 ', 'NC_082664.1 chromosome 4 ', 'NC_082665.1 chromosome 5 ', 'NC_082666.1 chromosome 6 ', 'NC_082667.1 chromosome 7 ', 'NC_082668.1 chromosome 8 ', 'NC_082669.1 chromosome 9 ', 'NC_082670.1 chromosome 10 ', 'NC_082671.1 chromosome 11 ', 'NC_082672.1 chromosome 12 ', 'NC_082673.1 chromosome 13 ', 'NC_082674.1 chromosome 14 ', 'NC_082675

{'labels': 0, 'species': 'Papio anubis', 'sampled_chromosomes': ['NC_044978.1 chromosome 3 ', 'NC_044984.1 chromosome 9 ', 'NC_044983.1 chromosome 8 ', 'NC_044991.1 chromosome 16 ', 'NC_044983.1 chromosome 8 ', 'NC_044976.1 chromosome 1 ', 'NC_044976.1 chromosome 1 ', 'NC_044997.1 chromosome Y ', 'NC_044991.1 chromosome 16 ', 'NC_044993.1 chromosome 18 '], 'chunks': ['AGGAAGGAAG', 'CTACTAGAAT', 'AATAGACAGA', 'GATACTCCTA', 'TATTTTGTCA', 'TAAAACAAAC', 'AAGGAACAGT', 'CCATTTTGCT', 'TCCAGCCTGT', 'AATATAGCAT'], 'full_sample_has_y_chr': True, 'has_y_chr_sampled': True, 'has_x_chr_sampled': False}
{'labels': 1, 'species': 'Bos javanicus', 'sampled_chromosomes': ['NC_083868.1 chromosome 1 ', 'NC_083875.1 chromosome 8 ', 'NC_083872.1 chromosome 5 ', 'NC_083870.1 chromosome 3 ', 'NC_083894.1 chromosome 27 ', 'NC_083868.1 chromosome 1 ', 'NC_083896.1 chromosome 29 ', 'NC_083872.1 chromosome 5 ', 'NC_083883.1 chromosome 16 ', 'NC_083879.1 chromosome 12 '], 'chunks': ['GGATAAAAAA', 'GCCTGCCATG', 'CG

DEBUG:mammals_gender_dataset:Label: 1
DEBUG:mammals_gender_dataset:Sample has Y chromosome: False
DEBUG:mammals_gender_dataset:Y chromosome in sampled chromosomes: False
DEBUG:mammals_gender_dataset:X chromosome in sampled chromosomes: False
DEBUG:mammals_gender_dataset:Sample organism name: Budorcas taxicolor
DEBUG:mammals_gender_dataset:Sample/assembly id: GCF_023091745.1
DEBUG:mammals_gender_dataset:Diploid: False
DEBUG:mammals_gender_dataset:Label: 1
DEBUG:mammals_gender_dataset:Sex chromosomes: ['NC_068935.1 chromosome X ']
DEBUG:mammals_gender_dataset:Autosomes: ['NC_068910.1 chromosome 1 ', 'NC_068911.1 chromosome 2 ', 'NC_068912.1 chromosome 3 ', 'NC_068913.1 chromosome 4 ', 'NC_068914.1 chromosome 5 ', 'NC_068915.1 chromosome 6 ', 'NC_068916.1 chromosome 7 ', 'NC_068917.1 chromosome 8 ', 'NC_068918.1 chromosome 9 ', 'NC_068919.1 chromosome 10 ', 'NC_068920.1 chromosome 11 ', 'NC_068921.1 chromosome 12 ', 'NC_068922.1 chromosome 13 ', 'NC_068923.1 chromosome 14 ', 'NC_068924.1 

{'labels': 1, 'species': 'Papio anubis', 'sampled_chromosomes': ['NC_044988.1 chromosome 13 ', 'NC_044988.1 chromosome 13 ', 'NC_044987.1 chromosome 12 ', 'NC_044994.1 chromosome 19 ', 'NC_044987.1 chromosome 12 ', 'NC_044984.1 chromosome 9 ', 'NC_044978.1 chromosome 3 ', 'NC_044980.1 chromosome 5 ', 'NC_044986.1 chromosome 11 ', 'NC_044982.1 chromosome 7 '], 'chunks': ['GATATATAAA', 'CTGAAGACAG', 'CTTGGTACAA', 'TGAACCAATT', 'GTAAGATCAA', 'GTTTGTACAG', 'ACAGTTAACT', 'CAAAGATTTA', 'ACCTCTACAT', 'GGCCTCTGGA'], 'full_sample_has_y_chr': False, 'has_y_chr_sampled': False, 'has_x_chr_sampled': False}
{'labels': 1, 'species': 'Budorcas taxicolor', 'sampled_chromosomes': ['NC_068922.1 chromosome 13 ', 'NC_068917.1 chromosome 8 ', 'NC_068911.1 chromosome 2 ', 'NC_068917.1 chromosome 8 ', 'NC_068913.1 chromosome 4 ', 'NC_068920.1 chromosome 11 ', 'NC_068925.1 chromosome 16 ', 'NC_068911.1 chromosome 2 ', 'NC_068910.1 chromosome 1 ', 'NC_068919.1 chromosome 10 '], 'chunks': ['TTTTTTTTTT', 'GAATCC

...

{'labels': 1, 'species': 'Macaca mulatta', 'sampled_chromosomes': ['NC_041773.1 chromosome 20 ', 'NC_041767.1 chromosome 14 ', 'NC_041767.1 chromosome 14 ', 'NC_041754.1 chromosome 1 ', 'NC_041758.1 chromosome 5 ', 'NC_041774.1 chromosome X ', 'NC_041755.1 chromosome 2 ', 'NC_041765.1 chromosome 12 ', 'NC_041760.1 chromosome 7 ', 'NC_041756.1 chromosome 3 '], 'chunks': ['ATGTGTGTGT', 'AACTTTCTCC', 'TTTTCCTCAA', 'GTCTCGATCT', 'ACGAGATCTG', 'TTACTTACAG', 'AAGCTGCTGT', 'TAATACTTTC', 'AATTACCTTG', 'TGGCTCTGTT'], 'full_sample_has_y_chr': False, 'has_y_chr_sampled': False, 'has_x_chr_sampled': True}
{'labels': 1, 'species': 'Mustela lutreola', 'sampled_chromosomes': ['NC_081291.1 chromosome 2 ', 'NC_081292.1 chromosome 3 ', 'NC_081304.1 chromosome 15 ', 'NC_081290.1 chromosome 1 ', 'NC_081294.1 chromosome 5 ', 'NC_081297.1 chromosome 8 ', 'NC_081297.1 chromosome 8 ', 'NC_081298.1 chromosome 9 ', 'NC_081296.1 chromosome 7 ', 'NC_081293.1 chromosome 4 '], 'chunks': ['GGAAAGGCTC', 'GGATTTTCCA',

DEBUG:mammals_gender_dataset:Label: 0.0
DEBUG:mammals_gender_dataset:Sample has Y chromosome: True
DEBUG:mammals_gender_dataset:Y chromosome in sampled chromosomes: True
DEBUG:mammals_gender_dataset:X chromosome in sampled chromosomes: True
DEBUG:mammals_gender_dataset:Sample organism name: Rattus rattus
DEBUG:mammals_gender_dataset:Sample/assembly id: GCF_011064425.1
DEBUG:mammals_gender_dataset:Diploid: False
DEBUG:mammals_gender_dataset:Label: 1
DEBUG:mammals_gender_dataset:Sex chromosomes: ['NC_046172.1 chromosome X ']
DEBUG:mammals_gender_dataset:Autosomes: ['NC_046154.1 chromosome 1 ', 'NC_046155.1 chromosome 2 ', 'NC_046156.1 chromosome 3 ', 'NC_046157.1 chromosome 4 ', 'NC_046158.1 chromosome 5 ', 'NC_046159.1 chromosome 6 ', 'NC_046160.1 chromosome 7 ', 'NC_046161.1 chromosome 8 ', 'NC_046162.1 chromosome 9 ', 'NC_046163.1 chromosome 10 ', 'NC_046164.1 chromosome 11 ', 'NC_046165.1 chromosome 12 ', 'NC_046166.1 chromosome 13 ', 'NC_046167.1 chromosome 14 ', 'NC_046168.1 chromo

{'labels': 0.0, 'species': 'Mus musculus', 'sampled_chromosomes': ['T+_Itpr3tf_J_chr15', 'T+_Itpr3tf_J_chr2', 'T+_Itpr3tf_J_chrY', 'T+_Itpr3tf_J_chr9', 'T+_Itpr3tf_J_chr7', 'T+_Itpr3tf_J_chr4', 'T+_Itpr3tf_J_chrX', 'T+_Itpr3tf_J_chr4', 'T+_Itpr3tf_J_chr3', 'T+_Itpr3tf_J_chr14'], 'chunks': ['CAATCCCAGA', 'ATACATAATA', 'TACATACCAT', 'TCCCATTTGT', 'GCCAACATCA', 'TTCTTTTCTT', 'CTCCCACAAA', 'AACCTTCCCA', 'ATTTCCATAC', 'CAGAGGCAGG'], 'full_sample_has_y_chr': True, 'has_y_chr_sampled': True, 'has_x_chr_sampled': True}
{'labels': 1, 'species': 'Rattus rattus', 'sampled_chromosomes': ['NC_046154.1 chromosome 1 ', 'NC_046158.1 chromosome 5 ', 'NC_046164.1 chromosome 11 ', 'NC_046159.1 chromosome 6 ', 'NC_046171.1 chromosome 18 ', 'NC_046167.1 chromosome 14 ', 'NC_046159.1 chromosome 6 ', 'NC_046161.1 chromosome 8 ', 'NC_046160.1 chromosome 7 ', 'NC_046161.1 chromosome 8 '], 'chunks': ['TCTTCTAGTG', 'TTCCACTGTC', 'TGTAAATGGC', 'TTTTCTGACC', 'GACAGATGGT', 'ATTTCAGGAT', 'GCTCACTGAA', 'AATCATAATT', 

DEBUG:mammals_gender_dataset:Label: 0
DEBUG:mammals_gender_dataset:Sample has Y chromosome: True
DEBUG:mammals_gender_dataset:Y chromosome in sampled chromosomes: True
DEBUG:mammals_gender_dataset:X chromosome in sampled chromosomes: False


{'labels': 0, 'species': 'Mustela erminea', 'sampled_chromosomes': ['NC_045614.1 chromosome 1 ', 'NC_045626.1 chromosome 13 ', 'NC_045615.1 chromosome 2 ', 'NC_045618.1 chromosome 5 ', 'NC_045615.1 chromosome 2 ', 'NC_045618.1 chromosome 5 ', 'NC_045626.1 chromosome 13 ', 'NC_045636.1 chromosome Y ', 'NC_045615.1 chromosome 2 ', 'NC_045627.1 chromosome 14 '], 'chunks': ['AAAAAAAGAG', 'ACAAGTAGAC', 'TGTGGTGAGC', 'CACTTATCTT', 'TATGGTTTTG', 'AATTACGTGA', 'TTCGAGCGAA', 'CCTGGATGAT', 'AATTGGATAT', 'TGTATGAATG'], 'full_sample_has_y_chr': True, 'has_y_chr_sampled': True, 'has_x_chr_sampled': False}
